# Wrist RGB grip-motion validation

Run the command in the KnowRob runbook to generate the motion cue report. This review compares the cues at the first and last sample to independent manual endpoint observations. The eight BNC clips reuse the Turtle review. Six further clips cover PICK and PLACE with gears, USB, pegs and the threaded piece. The manual labels are used during review only.

A cue indicates texture that stays fixed near the gripper while the scene moves in the wrist view. It leaves the object's identity and its relationship to a location undetermined.

In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

repo = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "main.py").is_file())
root = repo / "outputs/visual_reassemble_hand"
plan = json.loads((root / "plan.json").read_text(encoding="utf-8"))
report_path = repo / "outputs/bnc_state_inspection/rgb_grip_motion_validation.json"
report = json.loads(report_path.read_text(encoding="utf-8"))

# Independent visual review of the first and last saved samples, before cue comparison.
reference = {
    "2025-01-09-14-57-02_hand_clip_0058": ("unheld", "unheld"),  # Gear stays on the free surface.
    "2025-01-09-18-44-37_hand_clip_0060": ("held", "held"),  # Large gear remains in the jaws.
    "2025-01-09-18-08-36_hand_clip_0181": ("unheld", None),  # End view partly hides the grip and holder.
    "2025-01-09-14-40-21_hand_clip_0118": ("held", "held"),  # Round peg stays between the jaws.
    "2025-01-09-18-08-36_hand_clip_0071": ("unheld", None),  # End view partly hides the threaded piece.
    "2025-01-09-14-40-21_hand_clip_0199": ("held", "unheld"),  # Jaws open and withdraw from the peg.
}

bnc = (repo / "knowledge/observations/bnc_validation_window.ttl").read_text(encoding="utf-8")
bnc_endpoints = {}
for block in re.split(r"(?m)^observation:", bnc)[1:]:
    clip_id, endpoint = block.split(maxsplit=1)[0].rsplit("_", 1)
    values = dict(re.findall(r'reassemble:(\w+) "([^"]+)"', block))
    grip, = re.findall(r"reassemble:gripStatus reassemble:(\w+)", block)
    bnc_endpoints[clip_id, endpoint] = (grip, int(values["frameIndex"]), float(values["observedAt"]))

for clip_id, _ in bnc_endpoints:
    reference[clip_id] = (bnc_endpoints[clip_id, "first"][0], bnc_endpoints[clip_id, "last"][0])

In [ ]:
rows = report["observations"]
assert {row["clip_id"] for row in rows} == set(reference)
assert len(rows) == len(reference)
endpoint_results = Counter()
pair_results = Counter()
clips_with_cue = 0

for row in rows:
    clip_id = row["clip_id"]
    recording = row["recording_id"]
    assert recording in plan["split"]["validation"]
    directory = root / f"{recording}_hand"
    clips = json.loads((directory / "clips.json").read_text(encoding="utf-8"))
    clip, = [clip for clip in clips if clip["clip_id"] == clip_id]
    assert row["camera"] == clip["camera"] == "hand"
    assert row["segment_id"] == clip["segment_id"]
    indices, times = clip["sampled_frame_indices"], clip["original_frame_times"]
    pairs = row["intervals"]
    assert [(pair["first_frame"], pair["last_frame"], pair["first_time"], pair["last_time"])
            for pair in pairs] == list(zip(indices, indices[1:], times, times[1:]))
    if (clip_id, "first") in bnc_endpoints:
        for endpoint, slot in (("first", 0), ("last", -1)):
            _, index, time = bnc_endpoints[clip_id, endpoint]
            assert (index, time) == (indices[slot], times[slot])
    pair_results.update(pair["reason"] for pair in pairs)
    clips_with_cue += any(pair["cue"] for pair in pairs)
    for manual, pair in zip(reference[clip_id], (pairs[0], pairs[-1]), strict=True):
        endpoint_results[manual, pair["cue"] is not None] += 1

print("Paired samples:", sum(pair_results.values()))
print("Clips with at least one cue:", clips_with_cue, "/", len(rows))
print("Pair diagnostics:", dict(pair_results))
print("Endpoints by manual state and cue presence:", dict(endpoint_results))